# 📊 Análise de Sentimento no Mercado de Criptomoedas

**Disciplina:** Ciência de Dados  
**Professor:** Igor Junqueira  
**Instituição:** UNIUBE — Universidade de Uberaba  
**Ano:** 2026

---

## Objetivo

Este notebook documenta o pipeline completo de coleta, pré-processamento, análise de sentimento, clustering temático e correlação estatística aplicados ao mercado de criptomoedas.

**Pergunta de pesquisa:** *As opiniões expressas em portais de notícias sobre criptomoedas influenciam o preço do Bitcoin?*

---

## Etapas do Pipeline

| # | Etapa | Técnica |
|---|---|---|
| 1 | Coleta de dados | RSS Feeds + NewsAPI.org |
| 2 | Pré-processamento | Limpeza, deduplicação, detecção de entidades |
| 3 | Análise de Sentimento | VADER (Valence Aware Dictionary and sEntiment Reasoner) |
| 4 | Clustering Temático | K-Means + TF-IDF |
| 5 | Correlação Estatística | Coeficiente de Pearson (sentimento × preço BTC) |

---
## 0. Importações e Configuração

In [ ]:
import sqlite3
import os
import re
import warnings
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import yfinance as yf

warnings.filterwarnings('ignore')

# Estilo dos gráficos
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'DejaVu Sans',
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

DB_PATH = 'crypto_reddit.db'

print('✅ Bibliotecas carregadas com sucesso!')
print(f'📁 Banco de dados: {os.path.abspath(DB_PATH)}')

---
## 1. Coleta de Dados

Os dados foram coletados de duas fontes complementares:

**Fonte 1 — RSS Feeds (coleta contínua):**  
16 portais internacionais de notícias sobre criptomoedas monitorados automaticamente a cada 30 minutos via feeds RSS. Os portais incluem CoinDesk, CoinTelegraph, Decrypt, Bitcoin Magazine, entre outros.

**Fonte 2 — NewsAPI.org (dados históricos):**  
Importação em lote de artigos históricos utilizando 16 consultas temáticas distintas, resultando em 652 artigos únicos cobrindo o período de janeiro a junho de 2026.

A seguir, carregamos os dados armazenados no banco SQLite:

In [ ]:
# Carrega todos os dados do banco SQLite
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("""
    SELECT id, source_type, subreddit AS fonte, author, created_utc,
           text_content, crypto, famous_person,
           sentiment, sentiment_label, relevance_score
    FROM reddit_mentions
    ORDER BY created_utc DESC
""", conn)
conn.close()

df['created_utc'] = pd.to_datetime(df['created_utc'], utc=True, format='mixed')

print(f'📊 Total de registros carregados: {len(df):,}')
print(f'📅 Período: {df["created_utc"].min().date()} → {df["created_utc"].max().date()}')
print(f'🗂️  Fontes únicas: {df["fonte"].nunique()}')
print(f'🪙  Criptomoedas monitoradas: {sorted(df["crypto"].unique())}')
df.head()

In [ ]:
# Distribuição por tipo de fonte
print('📰 Publicações por tipo de fonte:')
print(df['source_type'].value_counts().to_string())

print('\n🪙 Publicações por criptomoeda:')
print(df['crypto'].value_counts().to_string())

---
## 2. Pré-processamento

O pré-processamento envolveu as seguintes etapas:

1. **Deduplicação** por URL/ID único (evita contar a mesma notícia duas vezes)
2. **Detecção de criptomoedas** via expressões regulares (Bitcoin, Ethereum, XRP, Solana, Dogecoin, Cardano, BNB)
3. **Detecção de personalidades** via correspondência de padrões (Elon Musk, Michael Saylor, Vitalik Buterin, Donald Trump, etc.)
4. **Filtragem de texto** — mínimo de 20 caracteres por publicação
5. **Normalização temporal** — conversão de todos os timestamps para UTC

In [ ]:
# Verifica qualidade dos dados
print('🔍 Verificação de qualidade dos dados:')
print(f'  Registros totais:         {len(df):,}')
print(f'  Valores nulos (sentimento): {df["sentiment"].isna().sum()}')
print(f'  Textos com < 20 chars:    {(df["text_content"].str.len() < 20).sum()}')
print(f'  Duplicatas por ID:        {df["id"].duplicated().sum()}')
print()

# Distribuição de comprimento dos textos
df['text_len'] = df['text_content'].str.len()
print('📏 Comprimento médio dos textos:')
print(f'  Mínimo:  {df["text_len"].min()} caracteres')
print(f'  Médio:   {df["text_len"].mean():.0f} caracteres')
print(f'  Máximo:  {df["text_len"].max()} caracteres')

In [ ]:
# Exemplo de detecção de entidades por regex
CRYPTO_KEYWORDS = {
    'bitcoin':  ['bitcoin', 'btc', 'satoshi'],
    'ethereum': ['ethereum', 'eth', 'ether'],
    'solana':   ['solana', 'sol'],
    'xrp':      ['xrp', 'ripple'],
    'dogecoin': ['dogecoin', 'doge'],
}

def detectar_crypto(texto: str) -> list:
    texto = f' {texto or ""} '.lower()
    encontradas = []
    for nome, keywords in CRYPTO_KEYWORDS.items():
        for kw in keywords:
            pattern = rf'(?<![a-z0-9]){re.escape(kw)}(?![a-z0-9])' if len(kw) <= 4 else kw
            if re.search(pattern, texto) if len(kw) <= 4 else kw in texto:
                encontradas.append(nome)
                break
    return encontradas

# Demonstração com exemplos reais
exemplos = [
    'Bitcoin price hits new ATH as ETH and BTC rally together',
    'Elon Musk tweets about Dogecoin again, DOGE pumps 30%',
    'XRP wins SEC lawsuit, Ripple celebrates regulatory victory',
]
print('🔎 Exemplos de detecção de criptomoedas:')
for ex in exemplos:
    print(f'  Texto: "{ex[:60]}..."')
    print(f'  Detectadas: {detectar_crypto(ex)}\n')

---
## 3. Análise de Sentimento com VADER

O **VADER** (Valence Aware Dictionary and sEntiment Reasoner) é um analisador de sentimento baseado em léxico e regras, desenvolvido especificamente para textos de mídias sociais e portais de notícias.

Ele retorna quatro escores para cada texto:
- `pos` — proporção positiva
- `neg` — proporção negativa  
- `neu` — proporção neutra
- `compound` — escore geral normalizado entre **-1** (mais negativo) e **+1** (mais positivo)

**Critério de classificação adotado:**
- `compound ≥ 0.05` → **Positivo**
- `compound ≤ -0.05` → **Negativo**
- entre os limiares → **Neutro**

In [ ]:
analyzer = SentimentIntensityAnalyzer()

# Demonstração do VADER com exemplos reais do corpus
exemplos_sentimento = [
    'Bitcoin is mooning again! The market is going crazy bullish right now! 🚀',
    'Crypto market crashed hard today. BTC down 15%, investors panicking.',
    'Bitcoin ETF approval expected next quarter according to analysts.',
    'Elon Musk just dumped all Bitcoin from Tesla. BTC price crashing.',
    'Michael Saylor says Bitcoin will reach $1 million by 2030.',
]

print('📊 Demonstração do VADER Sentiment Analysis:\n')
print(f'{"Texto":<55} {"Compound":>10} {"Rótulo":>10}')
print('-' * 80)
for texto in exemplos_sentimento:
    scores = analyzer.polarity_scores(texto)
    compound = scores['compound']
    rotulo = 'positivo' if compound >= 0.05 else ('negativo' if compound <= -0.05 else 'neutro')
    print(f'{texto[:54]:<55} {compound:>10.4f} {rotulo:>10}')

In [ ]:
# Estatísticas do sentimento no corpus completo
print('📈 Estatísticas de sentimento do corpus:')
print(f'  Sentimento médio (compound): {df["sentiment"].mean():.4f}')
print(f'  Desvio padrão:               {df["sentiment"].std():.4f}')
print(f'  Mínimo:                      {df["sentiment"].min():.4f}')
print(f'  Máximo:                      {df["sentiment"].max():.4f}')
print()

contagem = df['sentiment_label'].value_counts()
print('🏷️  Distribuição por rótulo:')
for label, count in contagem.items():
    pct = count / len(df) * 100
    print(f'  {label:<12}: {count:>4} publicações ({pct:.1f}%)')

In [ ]:
# Visualização: distribuição de sentimento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pizza
cores = ['#2ecc71', '#e74c3c', '#3498db']
contagem_plot = df['sentiment_label'].value_counts()
axes[0].pie(
    contagem_plot.values,
    labels=[f'{l.capitalize()}\n{v/len(df)*100:.1f}%' for l, v in contagem_plot.items()],
    colors=cores, startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
    textprops={'fontsize': 11, 'fontweight': 'bold'}
)
axes[0].set_title('Distribuição de Sentimento', fontsize=13, fontweight='bold', pad=15)

# Histograma do compound
axes[1].hist(df['sentiment'], bins=40, color='#3498db', edgecolor='white', alpha=0.8)
axes[1].axvline(0.05,  color='#2ecc71', linestyle='--', linewidth=1.5, label='Limiar positivo (0.05)')
axes[1].axvline(-0.05, color='#e74c3c', linestyle='--', linewidth=1.5, label='Limiar negativo (-0.05)')
axes[1].axvline(df['sentiment'].mean(), color='orange', linestyle='-', linewidth=2,
                label=f'Média ({df["sentiment"].mean():.3f})')
axes[1].set_xlabel('Escore Compound (VADER)', fontsize=11)
axes[1].set_ylabel('Frequência', fontsize=11)
axes[1].set_title('Distribuição do Escore VADER Compound', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Análise de Sentimento — Corpus Completo', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_sentimento.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: fig_sentimento.png')

In [ ]:
# Sentimento por criptomoeda
sent_crypto = df.groupby('crypto')['sentiment'].agg(['mean', 'count']).reset_index()
sent_crypto = sent_crypto[sent_crypto['count'] >= 5].sort_values('mean', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
cores_bar = ['#e74c3c' if v < 0 else '#2ecc71' for v in sent_crypto['mean']]
bars = ax.barh(sent_crypto['crypto'].str.upper(), sent_crypto['mean'], color=cores_bar, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
for bar, (_, row) in zip(bars, sent_crypto.iterrows()):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{row["mean"]:.3f} (n={int(row["count"])})', va='center', fontsize=9)
ax.set_xlabel('Sentimento Médio (VADER Compound)', fontsize=11)
ax.set_title('Sentimento Médio por Criptomoeda', fontsize=13, fontweight='bold')
ax.set_xlim(-0.3, 0.6)
plt.tight_layout()
plt.savefig('fig_sentimento_crypto.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Clustering Temático (K-Means + TF-IDF)

Para identificar automaticamente os principais **tópicos de discussão** no corpus, aplicamos:

1. **TF-IDF** (Term Frequency–Inverse Document Frequency): vetoriza os textos ponderando a importância de cada termo — termos muito comuns (como "bitcoin" ou "crypto") recebem menor peso; termos discriminativos recebem maior peso.

2. **K-Means**: algoritmo de clustering não supervisionado que agrupa publicações similares em *k* clusters, minimizando a variância intra-cluster.

O valor **k=5** foi definido como padrão, permitindo identificar 5 tópicos principais.

In [ ]:
# Prepara textos para clustering
textos = df['text_content'].fillna('').tolist()
N_CLUSTERS = 5

# Vetorização TF-IDF
vectorizer = TfidfVectorizer(
    max_features=500,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2
)
X_tfidf = vectorizer.fit_transform(textos)
print(f'✅ Matriz TF-IDF: {X_tfidf.shape[0]} documentos × {X_tfidf.shape[1]} termos')

# Clustering K-Means
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_tfidf)
df['cluster'] = labels
print(f'✅ K-Means aplicado com k={N_CLUSTERS} clusters')

# Extrai palavras-chave de cada cluster
feature_names = vectorizer.get_feature_names_out()
STOPWORDS_EXTRA = {'bitcoin', 'btc', 'crypto', 'cryptocurrency', 'coin', 'token',
                   'market', 'price', 'trading', 'said', 'new', 'one', 'also', 'get'}

print('\n🔍 Palavras-chave por tópico (TF-IDF):\n')
resultados_clusters = []
for i in range(N_CLUSTERS):
    center = kmeans.cluster_centers_[i]
    top_idx = center.argsort()[-15:][::-1]
    palavras = [feature_names[j] for j in top_idx if feature_names[j] not in STOPWORDS_EXTRA][:8]
    total = int((labels == i).sum())
    resultados_clusters.append({'Tópico': f'Tópico {i+1}', 'Palavras-chave': ', '.join(palavras), 'Posts': total})
    print(f'  Tópico {i+1} ({total} posts): {", ".join(palavras)}')

df_clusters = pd.DataFrame(resultados_clusters)
df_clusters

In [ ]:
# Sentimento por cluster
sent_cluster = df.groupby(['cluster', 'sentiment_label']).size().unstack(fill_value=0).reset_index()
sent_cluster['topico'] = sent_cluster['cluster'].apply(lambda x: f'Tópico {x+1}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras: total por tópico
total_por_cluster = df['cluster'].value_counts().sort_index()
cores_cluster = ['#9b59b6', '#f39c12', '#2c3e50', '#3498db', '#1abc9c']
axes[0].bar(
    [f'Tópico {i+1}' for i in range(N_CLUSTERS)],
    total_por_cluster.values,
    color=cores_cluster, edgecolor='white'
)
for i, v in enumerate(total_por_cluster.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Total de Publicações por Tópico', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Publicações')

# Barras: sentimento por tópico
x = np.arange(N_CLUSTERS)
w = 0.28
for col, cor, offset in [('positivo','#2ecc71',-w), ('neutro','#3498db',0), ('negativo','#e74c3c',w)]:
    if col in sent_cluster.columns:
        axes[1].bar(x + offset, sent_cluster[col], w, label=col.capitalize(), color=cor, edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Tópico {i+1}' for i in range(N_CLUSTERS)])
axes[1].set_title('Sentimento por Tópico', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Publicações')
axes[1].legend()

plt.suptitle('Clustering Temático K-Means + TF-IDF', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualização 2D dos clusters com PCA
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_tfidf.toarray())

fig, ax = plt.subplots(figsize=(9, 6))
for i, cor in enumerate(cores_cluster):
    mask = labels == i
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=cor, label=f'Tópico {i+1}',
               alpha=0.5, s=15, edgecolors='none')
ax.set_title('Visualização dos Clusters (PCA 2D)', fontsize=13, fontweight='bold')
ax.set_xlabel(f'Componente 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variância)')
ax.set_ylabel(f'Componente 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variância)')
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig('fig_pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Variância explicada pelos 2 componentes: {sum(pca.explained_variance_ratio_)*100:.1f}%')

---
## 5. Correlação: Sentimento × Preço do Bitcoin

Para responder à pergunta central da pesquisa, calculamos a **correlação de Pearson** entre:
- O **sentimento médio diário** das publicações coletadas
- A **variação percentual diária** do preço do Bitcoin (BTC-USD)

O preço histórico do BTC é obtido via biblioteca `yfinance` (Yahoo Finance).

**Hipótese:** dias com sentimento mais positivo nas notícias tendem a coincidir com dias de maior valorização do Bitcoin.

In [ ]:
# Sentimento médio diário
df['data'] = df['created_utc'].dt.date
sent_diario = df.groupby('data')['sentiment'].mean().reset_index()
sent_diario.columns = ['data', 'sentimento_medio']
sent_diario['data'] = pd.to_datetime(sent_diario['data'])

print(f'📅 Dias com publicações: {len(sent_diario)}')
print(f'📅 Período: {sent_diario["data"].min().date()} → {sent_diario["data"].max().date()}')

# Busca preço do BTC via yfinance
inicio = str(sent_diario['data'].min().date())
fim    = str((sent_diario['data'].max() + pd.Timedelta(days=1)).date())

print(f'\n🔄 Buscando preço BTC de {inicio} a {fim}...')
btc = yf.download('BTC-USD', start=inicio, end=fim, progress=False)
btc = btc[['Close']].reset_index()
btc.columns = ['data', 'preco_btc']
btc['data'] = pd.to_datetime(btc['data']).dt.tz_localize(None)
btc['variacao_pct'] = btc['preco_btc'].pct_change() * 100
print(f'✅ {len(btc)} dias de preço BTC carregados')
btc.tail()

In [ ]:
# Junta sentimento com preço do BTC
df_corr = sent_diario.merge(btc, on='data', how='inner').dropna()
print(f'📊 Dias com dados de sentimento E preço BTC: {len(df_corr)}')

# Calcula correlação de Pearson
r, p_valor = stats.pearsonr(df_corr['sentimento_medio'], df_corr['variacao_pct'])

print(f'\n📐 Resultado da Correlação de Pearson:')
print(f'  Coeficiente r : {r:.4f}')
print(f'  P-valor       : {p_valor:.4f}')
print(f'  Significativo : {"Sim (p < 0.05)" if p_valor < 0.05 else "Não (p >= 0.05)"}')
print(f'  Interpretação : correlação {"positiva" if r > 0 else "negativa"} e {"fraca" if abs(r) < 0.3 else "moderada" if abs(r) < 0.6 else "forte"}')

In [ ]:
# Visualizações da correlação
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
ax = axes[0]
ax.scatter(df_corr['sentimento_medio'], df_corr['variacao_pct'],
           color='#3498db', alpha=0.6, s=50, edgecolors='white')
m, b = np.polyfit(df_corr['sentimento_medio'], df_corr['variacao_pct'], 1)
xs = np.linspace(df_corr['sentimento_medio'].min(), df_corr['sentimento_medio'].max(), 100)
ax.plot(xs, m*xs+b, color='#e74c3c', linewidth=2, label=f'r = {r:.4f}  |  p = {p_valor:.4f}')
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Sentimento Médio Diário (VADER)', fontsize=11)
ax.set_ylabel('Variação do Preço BTC (%)', fontsize=11)
ax.set_title('Sentimento × Variação Diária do Bitcoin', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Série temporal duplo eixo
ax2 = axes[1]
ax3 = ax2.twinx()
ax2.plot(df_corr['data'], df_corr['sentimento_medio'],
         color='#3498db', linewidth=1.5, alpha=0.8, label='Sentimento Médio')
ax3.plot(df_corr['data'], df_corr['preco_btc'],
         color='#f39c12', linewidth=2, linestyle='--', label='Preço BTC (USD)')
ax2.set_ylabel('Sentimento Médio (VADER)', color='#3498db', fontsize=10)
ax3.set_ylabel('Preço BTC (USD)', color='#f39c12', fontsize=10)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v/1000:.0f}k'))
ax2.set_title('Sentimento vs Preço BTC ao Longo do Tempo', fontsize=12, fontweight='bold')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax3.get_legend_handles_labels()
ax2.legend(lines1+lines2, labels1+labels2, loc='upper right', fontsize=9)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30)

plt.suptitle('Correlação: Sentimento Midiático × Preço do Bitcoin', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Conclusões

### Resultados obtidos

| Métrica | Valor |
|---|---|
| Total de publicações analisadas | 974 |
| Sentimento médio VADER (compound) | 0,210 |
| Publicações positivas | 579 (59,4%) |
| Publicações negativas | 291 (29,9%) |
| Criptomoeda mais citada | Bitcoin (~52%) |
| Personalidade mais mencionada | Michael Saylor |
| Tópicos identificados (K-Means) | 5 |
| Correlação de Pearson (r) | 0,1934 |
| P-valor | 0,1928 |

### Interpretação

1. **Sentimento predominantemente positivo (59,4%)**: A mídia especializada em criptomoedas mantém viés otimista mesmo em períodos de queda de preços. Durante o período analisado, o Bitcoin recuou de ~US$90.000 para ~US$65.000, mas o sentimento médio permaneceu positivo (0,210).

2. **Clustering revelou 5 tópicos coerentes**: Elon Musk/SpaceX IPO, Mercado geral/ETFs, MicroStrategy/Saylor, Vitalik/Ethereum e Geopolítica/Trump — refletindo os temas dominantes na mídia cripto no período.

3. **Correlação de Pearson r = 0,193 (p = 0,193)**: A correlação é **positiva** (dias com sentimento mais alto tendem a coincidir com dias de maior valorização do BTC), porém **fraca e estatisticamente não significativa** (p > 0,05).

### Limitação e trabalhos futuros

- O VADER foi otimizado para redes sociais; modelos como **FinBERT** podem capturar melhor nuances do jornalismo financeiro
- Ampliar o horizonte temporal para 12+ meses e incluir defasagem temporal (lag) na análise de correlação
- Incorporar análise de **causalidade de Granger** para investigar precedência temporal
- Expandir para fontes em **português** e incluir dados de redes sociais

In [ ]:
# Resumo final
print('=' * 55)
print('  RESUMO FINAL — ANÁLISE DE SENTIMENTO CRIPTO')
print('=' * 55)
print(f'  Total de publicações:          {len(df):>6,}')
print(f'  Sentimento médio (VADER):      {df["sentiment"].mean():>8.4f}')
print(f'  Publicações positivas:         {(df["sentiment_label"]=="positivo").sum():>6,} ({(df["sentiment_label"]=="positivo").mean()*100:.1f}%)')
print(f'  Publicações negativas:         {(df["sentiment_label"]=="negativo").sum():>6,} ({(df["sentiment_label"]=="negativo").mean()*100:.1f}%)')
print(f'  Crypto mais citada:            {df["crypto"].value_counts().idxmax().upper():>8}')
print(f'  Tópicos K-Means:               {N_CLUSTERS:>6}')
print(f'  Correlação Pearson (r):        {r:>8.4f}')
print(f'  P-valor:                       {p_valor:>8.4f}')
print(f'  Significância (p<0.05):        {"Sim" if p_valor < 0.05 else "Não":>8}')
print('=' * 55)